# 3.1 · Feature engineering y baseline lineal

**Tiempo estimado:** 30 min.

**Objetivos.**

1. Construir la matriz de features para el caudal del Genil (lags + rolling + calendario + Fourier + lluvia retardada).
2. Implementar baselines: persistencia, persistencia estacional, climatología.
3. Ajustar una regresión Ridge con `skforecast` y comparar.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import Ridge
from skforecast.recursive import ForecasterRecursive

from cst import datos as ud

plt.rcParams.update({"figure.figsize": (10, 3.4), "axes.grid": True, "grid.alpha": 0.3})

## 1 · Datos

Trabajamos a frecuencia **diaria** — donde el ML brilla frente a los modelos estadísticos.

In [ ]:
caudal = ud.cargar_caudal_genil(source="CEDEX")
lluvia = ud.cargar_lluvia_genil_diaria(fecha_inicio="2010-01-01", fecha_fin="2020-12-31")

# Periodo 2011-2020 (10 años con cobertura diaria continua del SAIH).
# Interpolamos los pocos gaps cortos que quedan (≤7 días) para tener freq='D' limpia.
df = pd.DataFrame({"caudal": caudal, "lluvia": lluvia}).loc["2011":"2020"]
df = df.asfreq("D").interpolate("linear", limit=7).dropna()
print(f"Días: {len(df):,}   Freq: {df.index.freq}   NaN: {df.isna().sum().sum()}")
print(df.describe().round(2))

## 2 · Matriz de features

Función reutilizable. Importante: **rolling siempre `center=False`** y aplicado en `shift(1)` para que no haya fuga del futuro.

In [ ]:
def construir_features(df, target="caudal", exog="lluvia"):
    """Devuelve X (DataFrame) e y (Series) listos para fit/predict."""
    out = pd.DataFrame(index=df.index)

    # 1) Lags del propio caudal
    for lag in (1, 2, 3, 5, 7, 14, 30, 90, 365):
        out[f"q_lag{lag}"] = df[target].shift(lag)

    # 2) Rolling del caudal (siempre con shift previo)
    for w in (7, 30, 90):
        out[f"q_mean{w}"] = df[target].shift(1).rolling(w).mean()
        out[f"q_max{w}"] = df[target].shift(1).rolling(w).max()
        out[f"q_std{w}"] = df[target].shift(1).rolling(w).std()

    # 3) Lluvia retardada y acumulada
    for lag in (1, 2, 3, 7):
        out[f"p_lag{lag}"] = df[exog].shift(lag)
    for w in (3, 7, 14, 30):
        out[f"p_acum{w}d"] = df[exog].rolling(w).sum().shift(1)

    # 4) Calendario + Fourier
    idx = out.index
    out["mes"] = idx.month
    out["sin_an"] = np.sin(2 * np.pi * idx.dayofyear / 365.25)
    out["cos_an"] = np.cos(2 * np.pi * idx.dayofyear / 365.25)

    # 5) Target
    out["y"] = df[target]

    # dropna() pierde el atributo freq → lo reconstruimos con asfreq('D')
    return out.dropna().asfreq("D")


feat = construir_features(df)
print(f"Features: {feat.shape[1] - 1}   Filas válidas: {len(feat):,}   Freq: {feat.index.freq}")
feat.tail(3).round(2)

## 3 · Split temporal

Train hasta 2017, test 2018-2020.

In [ ]:
split = pd.Timestamp("2018-01-01")
train = feat.loc[:split].iloc[:-1]
test = feat.loc[split:]

y_train, y_test = train["y"], test["y"]
X_train = train.drop(columns="y")
X_test = test.drop(columns="y")
print(f"Train: {len(train):,} obs   Test: {len(test):,} obs   Features: {X_train.shape[1]}")

## 4 · Baselines

Tres que un modelo serio debe batir:

In [ ]:
def rmse(y, yhat):
    return float(np.sqrt(((y - yhat) ** 2).mean()))


def mae(y, yhat):
    return float((y - yhat).abs().mean())


# Baseline 1: persistencia (yhat = y_{t-1})
pred_pers = X_test["q_lag1"]

# Baseline 2: persistencia anual (yhat = y_{t-365})
pred_pers_anual = X_test["q_lag365"]

# Baseline 3: climatología (media histórica del día del año)
clim = y_train.groupby(y_train.index.dayofyear).mean()
pred_clim = y_test.index.dayofyear.map(clim)

for nombre, pred in [
    ("Persistencia (lag-1)", pred_pers),
    ("Persistencia anual (lag-365)", pred_pers_anual),
    ("Climatología (doy)", pred_clim),
]:
    print(f"{nombre:35s} RMSE={rmse(y_test, pred):.3f}  MAE={mae(y_test, pred):.3f}")

## 5 · Ridge con skforecast

`ForecasterRecursive` toma un regresor sklearn y lo convierte en forecaster multi-paso vía recursive.

**Nota:** skforecast espera que las features sean **autoregresivas** (lags del target) más exógenas. La columna `q_lag*` la maneja internamente; nosotros le pasamos sólo lluvia y calendario como exógenas.

In [ ]:
exog_cols = [c for c in X_train.columns if not c.startswith("q_")]
y_train_s = pd.Series(y_train.values, index=y_train.index, name="caudal")
y_test_s = pd.Series(y_test.values, index=y_test.index, name="caudal")
exog_train_df = X_train[exog_cols]
exog_test_df = X_test[exog_cols]

forecaster = ForecasterRecursive(
    regressor=Ridge(alpha=1.0),
    lags=[1, 2, 3, 7, 14, 30, 90, 365],
)
forecaster.fit(y=y_train_s, exog=exog_train_df)

# Predicción 1 día (rolling): usamos el último valor real disponible
pred_ridge = forecaster.predict(steps=len(y_test_s), exog=exog_test_df)

print(
    f"Ridge skforecast (recursivo h={len(y_test_s)})  "
    f"RMSE={rmse(y_test, pred_ridge):.3f}  MAE={mae(y_test, pred_ridge):.3f}"
)

**Cuidado:** este Ridge predice **toda la cola** recursivamente, alimentándose de sus propias predicciones. Para evaluar h=1 reajustado en cada paso, se usa `backtesting_forecaster` (notebook 03).

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(y_test.index, y_test.values, color="black", lw=1, label="observado")
ax.plot(y_test.index, pred_pers.values, color="#999", lw=0.8, ls=":", label="persistencia")
ax.plot(y_test.index, pred_ridge.values, color="#c2410c", lw=1, ls="--", label="Ridge skforecast")
ax.set_ylabel("Q (m³/s)")
ax.legend()
ax.set_title("Test 2018-2020 — predicción 1 día (con feedback)")
plt.tight_layout()

## 6 · Coeficientes — qué aprende Ridge

In [ ]:
# Acceder al modelo subyacente y a los nombres de features
modelo = forecaster.regressor
nombres = list(forecaster.lags_names) + list(exog_cols)
coef = pd.Series(modelo.coef_, index=nombres).sort_values(key=abs, ascending=False)
print(coef.head(15).round(4))

**Lectura típica:** el coeficiente de `lag_1` domina (el día anterior es el mejor predictor); siguen los lags cortos y la lluvia acumulada reciente. Que `lag_365` no sea cero confirma estacionalidad anual.

## 7 · Ejercicios

1. **Lasso.** Reajusta con `Lasso(alpha=0.1)`. ¿Qué features mantiene? ¿Es mejor o peor que Ridge?
2. **Ablation de features.** Entrena 3 variantes: sólo lags, lags + lluvia, todo. Compara RMSE.
3. **Año Fourier de orden superior.** Añade $\sin(4\pi t/365)$ y $\cos(4\pi t/365)$. ¿Captura mejor el ciclo?
4. **Polinómicas.** Añade `q_lag1 ** 2` y `q_lag1 * p_acum7d`. ¿Mejora la captura de picos?
5. **Reto.** Construye un `ForecasterRecursive` con `lags=[1..30]` y predict de h=7. Compara contra `ForecasterDirect` con la misma config.